# Example 03 — Ask questions and explore the KG

Pre-req: example 02 has finished and the Library has at least one indexed paper.

In [ ]:
import json

import httpx

BASE = 'http://localhost:8000'
LIBRARY = 'demo'
client = httpx.Client(base_url=BASE, timeout=120)

## One-shot QA

In [ ]:
res = client.post(
    f'/v1/libraries/{LIBRARY}/qa',
    json={'question': 'What is GraphRAG and how does it differ from vanilla RAG?'},
)
data = res.json()
print(data['answer'])
print('\nCitations:')
for c in data['citations']:
    print(f"  [{c['chunk_id']}] doc={c['doc_id']} page={c['page']}")
print(f"\nmodel={data['model']}  duration_ms={data['duration_ms']}")

## Streaming QA (SSE)

In [ ]:
with client.stream('GET', f'/v1/libraries/{LIBRARY}/qa/stream', params={'question': 'Summarize the main contributions.'}) as response:
    event = None
    for raw in response.iter_lines():
        line = raw if isinstance(raw, str) else raw.decode('utf-8')
        if line.startswith('event: '):
            event = line[7:]
        elif line.startswith('data: '):
            payload = json.loads(line[6:])
            print(f"[{event}] {payload}")

## KG neighborhood

In [ ]:
res = client.get(f'/v1/libraries/{LIBRARY}/entities/GraphRAG/neighborhood', params={'depth': 2})
for t in res.json()['triples']:
    print(f"  ({t['head']}) -[{t['relation']}]-> ({t['tail']})  conf={t['confidence']:.2f}  evidence={t['evidence_count']}")

## Generate a literature review

In [ ]:
res = client.post(
    f'/v1/libraries/{LIBRARY}/review',
    json={'topic': 'How does query routing affect retrieval quality in RAG systems?'},
)
review = res.json()
print('# ' + review['topic'])
if review.get('abstract'):
    print('\n' + review['abstract'])
for s in review['sections']:
    print('\n## ' + s['heading'])
    print(s['body'])